# Gemaakt door Jens Hooijmans & Ivar Hekkink

De **MLSP 2013 Bird Classification Challenge** draait om het automatisch herkennen van vogelsoorten in 10-seconden audio-opnames die in echte natuurcondities zijn gemaakt. Elke opname kan meerdere vogelsoorten bevatten (multi-label classificatie) en bevat vaak achtergrondgeluiden zoals wind, regen en insecten. De dataset bevat opnames uit een bosgebied met in totaal 19 vogelsoorten. De uitdaging is om de geluiden om te zetten in bruikbare kenmerken, bijvoorbeeld via spectrogrammen, en vervolgens een model te trainen dat voor elke soort voorspelt of die in de opname aanwezig is.


In [5]:
#%pip install librosa
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from tqdm.auto import tqdm

In [6]:

# --- Dynamische Pad Configuratie ---

# 1. Definieer de twee mogelijke paden
# Let op de r"..." voor het Windows-pad, dit is belangrijk voor de backslashes!
pad_crabbedfish = Path("/home/crabbedfish/Desktop/mlsp-2013-birds/mlsp_contest_dataset/mlsp_contest_dataset/essential_data/src_wavs")
pad_ivar = Path(r"C:\Users\Ivar\Desktop\Bird Data\mlsp-2013-birds\mlsp_contest_dataset\mlsp_contest_dataset\essential_data\src_wavs")

# 2. Controleer welk pad bestaat en stel 'data_map' in
if pad_crabbedfish.exists():
    data_map = pad_crabbedfish
    print("✅ Pad ingesteld voor: crabbedfish (Linux)")
elif pad_ivar.exists():
    data_map = pad_ivar
    print("✅ Pad ingesteld voor: Ivar (Windows)")
else:
    # Dit gebeurt als geen van beide paden wordt gevonden
    data_map = None
    print("❌ FOUT: Kon de data-map op geen van beide locaties vinden.")
    print(f"   Gezocht op (Linux): {pad_crabbedfish}")
    print(f"   Gezocht op (Windows): {pad_ivar}")

# --- Vanaf hier kun je de code gebruiken zoals voorheen ---

if data_map:
    print(f"\nWerken met map: {data_map}")
    
    # 3. Haal alle .wav-bestanden op uit de gevonden map
    wav_bestanden = list(data_map.glob("*.wav"))
    
    print(f"Aantal .wav bestanden gevonden: {len(wav_bestanden)}")
    
    # 4. Toon de eerste 5 bestanden als voorbeeld
    if wav_bestanden:
        print("\n--- Eerste 5 bestanden ---")
        for f in wav_bestanden[:5]:
            print(f.name)

✅ Pad ingesteld voor: crabbedfish (Linux)

Werken met map: /home/crabbedfish/Desktop/mlsp-2013-birds/mlsp_contest_dataset/mlsp_contest_dataset/essential_data/src_wavs
Aantal .wav bestanden gevonden: 645

--- Eerste 5 bestanden ---
PC8_20090705_050000_0030.wav
PC11_20100513_043000_0740.wav
PC15_20090513_070000_0030.wav
PC5_20090804_070000_0040.wav
PC16_20090705_070000_0040.wav


In [7]:

# Zorg dat 'wav_bestanden' en 'data_map' bestaan uit de vorige cellen
if 'wav_bestanden' not in locals() or not wav_bestanden:
    print("Run eerst de vorige cel om 'wav_bestanden' te vullen.")
else:
    # --- 1. Definieer en maak de output map ---
    
    # Weet waar je notebook is. 
    # Path.cwd() is de "Current Working Directory"
    # (Normaal gesproken /home/crabbedfish/Documents/GitHub/DataScienceProjectweek2025/)
    huidige_map = Path.cwd() 
    
    # Maak een nieuwe map genaamd 'spectrograms' in die map
    output_dir = huidige_map / "spectrograms"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Alle afbeeldingen worden opgeslagen in: {output_dir}")

    # --- Constantes voor het genereren ---
    SAMPLE_RATE = 22050
    N_MELS = 128 # 128 frequentiebanden (hoogte van de afbeelding)

    # --- 2. Loop door ALLE bestanden ---
    
    # tqdm() wikkelt zich om je lijst en toont een voortgangsbalk
    for wav_file in tqdm(wav_bestanden, desc="Spectrograms opslaan"):
        try:
            # --- 3. Genereren (zelfde als voorheen) ---
            y, sr = librosa.load(wav_file, sr=SAMPLE_RATE)
            
            # Sla over als het bestand te kort is (bijv. leeg)
            if len(y) < 512: # Minimaal 1 frame
                print(f"Bestand {wav_file.name} is te kort, wordt overgeslagen.")
                continue

            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            S_db = librosa.power_to_db(S, ref=np.max)

            # --- 4. Opslaan als 'kale' afbeelding ---
            
            # Maak een nieuwe, lege figuur
            fig = plt.figure(figsize=(10, 4)) # Grootte is niet super belangrijk, we croppen toch
            
            # Teken het spectrogram
            librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel')
            
            # VERWIJDER alle assen, labels, en witranden
            plt.axis('off')
            plt.tight_layout(pad=0)
            
            # Bepaal de naam van het output-bestand
            # We gebruiken .stem om de extensie (.wav) eraf te halen
            output_filename = output_dir / (wav_file.stem + ".png")
            
            # Sla op (bbox_inches='tight' en pad_inches=0 zijn cruciaal voor 'croppen')
            plt.savefig(output_filename, bbox_inches='tight', pad_inches=0)

            # --- 5. Sluit de figuur (BELANGRIJK!) ---
            # Dit voorkomt dat de plot wordt getoond en bespaart geheugen
            plt.close(fig)

        except Exception as e:
            # Vang eventuele fouten op (bijv. corrupte bestanden)
            print(f"Fout bij verwerken van {wav_file.name}: {e}")

    print("\n✅ Verwerking voltooid!")
    print(f"Alle spectrogrammen zijn opgeslagen in de map 'spectrograms'.")

Alle afbeeldingen worden opgeslagen in: /home/crabbedfish/Documents/GitHub/DataScienceProjectweek2025/spectrograms


Spectrograms opslaan:   0%|          | 0/645 [00:00<?, ?it/s]


✅ Verwerking voltooid!
Alle spectrogrammen zijn opgeslagen in de map 'spectrograms'.
